# Sensor Model V1 — Optical Flow-Through Sensor

Modelling the sensor signal from a known SPION concentration in the channel.
Basis: Beer-Lambert law, base model (single beam).

**Workflow:** calibration (real measurement data) → recalibration check → forward model → application

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import sys, os

sys.path.insert(0, os.path.abspath('..'))
from spektral_sensor.model.sensor_model import (
    beer_lambert, fit_beta, recalibrate_beta, sensor_signal, R_I, R_A
)

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

## 1. Geometry and parameters

Coordinate system: y-axis along the light path (light source → sensor), z-axis along the channel axis.

- Tubing: inner radius $r_i$, outer radius $r_a$
- $\alpha_S$: silicone absorption coefficient [1/mm]
- $\beta$: SPION absorption proportionality factor [mL/(mg·mm)]

In [ ]:
# Geometry (imported from the package)
r_i = R_I    # mm — inner radius of tubing
r_a = R_A    # mm — outer radius of tubing

# Optical parameters
alpha_S = 2.0    # 1/mm — silicone absorption coefficient

print(f'r_i = {r_i:.3f} mm')
print(f'r_a = {r_a:.3f} mm')
print(f'wall thickness = {r_a - r_i:.3f} mm')
print(f'optical path through suspension = 2*r_i = {2*r_i:.3f} mm')

## 2. Calibration — determination of $\beta$

**Model (base model, single beam):**
$$I_\text{sensor} = I_0 \cdot e^{-2\alpha_S(r_a - r_i)} \cdot e^{-\beta C}$$

Normalisation to the baseline ($c=0$, H₂O) cancels the constant silicone term:
$$\frac{I_\text{sensor}}{I_0} = e^{-\beta \cdot 2r_i \cdot c}
\quad \Rightarrow \quad
\ln\!\left(\frac{I_\text{sensor}}{I_0}\right) = -\beta \cdot 2r_i \cdot c$$

**Procedure:** take logarithm → linear fit → slope yields $\beta$.

This calibration is the basis for everything that follows — $\beta$ is fitted from real
measurement data, not assumed.

In [ ]:
# Calibration concentrations
c_pct = np.array([0, 2.5, 5, 10, 25, 50, 75, 100])   # % of 5 mg/mL stock
c = c_pct / 100 * 5.0                                 # mg/mL

# Measured values (mean of 3 replicates) [ADC counts], 640 nm
s0 = np.array([6175.75, 5095.25, 4450.42, 3109.02, 1277.47, 349.67, 158.51,  87.77])
s1 = np.array([6807.27, 5678.24, 4970.30, 3550.68, 1467.78, 419.97, 195.67, 112.47])
s2 = np.array([5772.13, 4826.33, 4245.17, 3053.35, 1272.14, 376.43, 173.69, 100.66])

sensors = [('Sensor 0', s0), ('Sensor 1', s1), ('Sensor 2', s2)]

betas = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for name, s in sensors:
    I_0_cal = s[0]              # baseline at c=0 (H2O)
    I_norm = s / I_0_cal        # I_sensor / I_0
    log_I_norm = np.log(I_norm)

    beta_s, r_sq = fit_beta(c, s, I_0_cal)
    betas[name] = beta_s
    slope = -beta_s * 2 * r_i

    print(f'{name}: beta = {beta_s:.4f} mL/(mg·mm),  R² = {r_sq:.4f}')

    # Left: raw signal
    axes[0].plot(c, s, 'o-', label=name)
    # Right: log-normalized + fit
    p = axes[1].plot(c, log_I_norm, 'o', label=f'{name} data')
    axes[1].plot(c, slope * c, '--', color=p[0].get_color(),
                 label=f'{name} fit (β={beta_s:.3f})')

axes[0].set_xlabel('c [mg/mL]')
axes[0].set_ylabel('$I_\\mathrm{sensor}$ [ADC]')
axes[0].set_title('Raw calibration signal')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('c [mg/mL]')
axes[1].set_ylabel(r'$\ln(I_\mathrm{sensor}/I_0)$')
axes[1].set_title('Beer-Lambert fit (intercept fixed at 0)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

beta = float(np.mean(list(betas.values())))
beta_std = float(np.std(list(betas.values())))
print(f'\nbeta (mean over all sensors): {beta:.4f} ± {beta_std:.4f} mL/(mg·mm)')

### 2b. Fit residuals

R² ≈ 0.97 indicates systematic deviation from strict log-linearity — expected at high
concentration (multiple scattering, detector floor). The residuals show where the base
model breaks down.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for name, s in sensors:
    log_I_norm = np.log(s / s[0])
    predicted = -betas[name] * 2 * r_i * c
    ax.plot(c, log_I_norm - predicted, 'o-', label=name)

ax.axhline(0, color='gray', linestyle='--', alpha=0.6)
ax.set_xlabel('c [mg/mL]')
ax.set_ylabel('Residual $\\ln(I/I_0) - $ fit')
ax.set_title('Calibration fit residuals')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Recalibration

In case of sensor drift: re-determine the baseline $I_0$ with H₂O before each measurement.
$\beta$ stays constant (from the full calibration above).

**Error of 2-point vs. full calibration:** expected < 6% — acceptable for a first model.

In [ ]:
print('2-point recalibration (c=0 and c=5.0 mg/mL) vs. full calibration:\n')

for i, (name, s) in enumerate(sensors):
    beta_2pt = recalibrate_beta(s[0], s[-1], c_100=5.0, r_i=r_i)
    dev = abs(beta_2pt - beta) / beta * 100
    print(f'{name}: beta_2pt = {beta_2pt:.4f},  '
          f'beta_full = {betas[name]:.4f},  '
          f'deviation from mean = {dev:.1f}%')

## 4. Sensor model — forward model

**Base model (single beam, x=0):**
$$I_\text{sensor}(t) = I_0 \cdot e^{-\beta C(t)}$$

with the cumulative concentration along the light beam:
$$C(t) = 2\int_0^{r_i} c(r,\, t)\, dr$$

**Input:** $c(r, t)$ — radial concentration profile as a 2D array `[Nr, Nt]`  
**Output:** $I_\text{sensor}(t)$ — sensor signal as a time vector

Implemented as `sensor_signal(c_rt, r, beta, I_0)` in `spektral_sensor.model.sensor_model`.

In [ ]:
help(sensor_signal)

## 5. Example application — synthetic bolus

Testing the model with a synthetic bolus:
- Gaussian in $t$, uniform in $r$
- Placeholder for $c(r,t)$ from the flow model

$\beta$ from the calibration above, $I_0$ per sensor from its own baseline.

In [ ]:
# Grid
Nt, Nr = 200, 50
t_max = 10.0                        # s
t = np.linspace(0, t_max, Nt)       # s
r = np.linspace(0, r_i, Nr)         # mm

# Synthetic bolus: Gaussian in t, uniform in r
t_center = t_max / 2   # s  — bolus centre
sigma_t = 1.0          # s  — bolus width
c_peak = 5.0           # mg/mL

c_rt = c_peak * np.exp(-(t[None, :] - t_center)**2 / (2 * sigma_t**2)) \
       * np.ones((Nr, 1))           # [Nr, Nt]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Cumulative concentration
C_t = 2 * np.trapezoid(c_rt, r, axis=0)
axes[0].plot(t, C_t)
axes[0].set_xlabel('t [s]')
axes[0].set_ylabel('C(t) [mg/mL · mm]')
axes[0].set_title('Cumulative concentration C(t)')
axes[0].grid(True, alpha=0.3)

# Sensor signals — each sensor with its own baseline, shared beta
for name, s in sensors:
    I_t = sensor_signal(c_rt, r, beta, s[0])
    p = axes[1].plot(t, I_t, label=name)
    axes[1].axhline(s[0], color=p[0].get_color(), linestyle='--', alpha=0.4)

axes[1].set_xlabel('t [s]')
axes[1].set_ylabel('$I_\\mathrm{sensor}$ [ADC]')
axes[1].set_title('Sensor signal — synthetic bolus\n(dashed: baselines)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'C_max = {C_t.max():.2f} mg/mL·mm')
print(f'Signal drop at peak: {(1 - np.exp(-beta * C_t.max())) * 100:.1f}% of baseline')

## 6. Application to simulation data (E02_20x15)

### Baseline and 100% reference

| Sample | $I_{\mathrm{Baseline},A}$ / ADC | $I_{\mathrm{Baseline},B}$ / ADC | $I_{\mathrm{Baseline},\mathrm{Ref}}$ / ADC |
|---|---:|---:|---:|
| `E02_50R_AB_1` | 7616 | 5492 | 6129 |

### 100% reference values

| Channel | Sensor | $I_{100\%}$ / ADC | Time of minimum / s |
|---|---:|---:|---:|
| A | 1001 | 114 | 9.902 |
| B | 1000 | 69 | 9.829 |
| Ref | 1002 | 63 | 4.103 |

$I_{100\%}$ is taken as the absolute minimum after the reference injection.
Per-sensor $\beta$ is obtained by 2-point recalibration from these values.

In [ ]:
# Per-sensor baseline and 100% reference for E02
E02 = {
    'A   (1001)': dict(I_0=7616.0, I_100=114.0),
    'B   (1000)': dict(I_0=5492.0, I_100= 69.0),
    'Ref (1002)': dict(I_0=6129.0, I_100= 63.0),
}

print('2-point recalibrated beta per sensor (E02):\n')
for name, v in E02.items():
    v['beta'] = recalibrate_beta(v['I_0'], v['I_100'], c_100=5.0, r_i=r_i)
    dev = (v['beta'] - beta) / beta * 100
    print(f"{name}: beta = {v['beta']:.4f} mL/(mg·mm)   "
          f"({dev:+.1f}% vs. calibration mean {beta:.4f})")

### Loading $c(r,t)$ from the flow simulation

Expected format: CSV with rows = radial positions, columns = time steps.
Format to be agreed with the flow-simulation partner.

**Open question:** the CSV carries no time axis — `t_max` (or the time step `dt`)
must be supplied as metadata by the flow model. Placeholder below.

In [ ]:
import pandas as pd
from pathlib import Path

SIM_FILE = Path('c_sim.csv')

if not SIM_FILE.exists():
    print(f'{SIM_FILE} not found — skipping. '
          'Place the simulation output here to run this section.')
else:
    c_sim = pd.read_csv(SIM_FILE, index_col=0)
    Nr_sim, Nt_sim = c_sim.shape
    c_rt_sim = c_sim.values

    # TODO: t_max must come from the flow model as metadata
    t_max_sim = 10.0   # s — PLACEHOLDER

    r_sim = np.linspace(0, r_i, Nr_sim)      # mm
    t_sim = np.linspace(0, t_max_sim, Nt_sim)  # s

    print(f'Loaded c(r,t): Nr={Nr_sim}, Nt={Nt_sim}')
    print(f'c range: {c_rt_sim.min():.3f} .. {c_rt_sim.max():.3f} mg/mL')

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for name, v in E02.items():
        I_sim = sensor_signal(c_rt_sim, r_sim, v['beta'], v['I_0'])
        p = ax.plot(t_sim, I_sim, label=name)
        ax.axhline(v['I_0'], color=p[0].get_color(), linestyle='--', alpha=0.4)

    ax.set_xlabel('t [s]')
    ax.set_ylabel('$I_\\mathrm{sensor}$ [ADC]')
    ax.set_title('Modelled sensor signal from simulated c(r,t) — E02')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Summary

| Step | Result |
|---|---|
| Full calibration (8 points, 3 sensors) | β = 0.271 ± 0.005 mL/(mg·mm), R² ≈ 0.974 |
| 2-point recalibration error | < 6% — acceptable for the first model |
| Forward model | `sensor_signal(c_rt, r, beta, I_0)` verified against an analytic synthetic bolus |

**Open points**

- R² ≈ 0.974 rather than > 0.99: residuals grow at high $c$ — the base model neglects
  multiple scattering and the detector noise floor. A single-beam Beer-Lambert model may
  need a scattering correction term.
- The simulation CSV needs an explicit time axis (or `dt`) as metadata.
- Extension from single beam (x=0) to the full beam geometry across the channel width.